### Import Libraries

In [7]:
import os
import json
import pandas as pd
from pathlib import Path

from product_search.evaluation import SearchEvaluator
from product_search.benchmarking import (
    retrieve_bm25,
    retrieve_hnsw,
    retrieve_hybrid_rrf,
    retrieve_hybrid_rerank,
)
from product_search.search_pipeline import OpenSearchInference

project_dir = Path(os.getcwd()).parent
data_dir = project_dir / "Data" / "RAW"
processed_dir = project_dir / "Data" / "PROCESSED"

In [8]:
# Load dev.env into os.environ (setdefault so explicit env vars take precedence)
_env_file = project_dir / "dev.env"
if _env_file.exists():
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _key, _, _val = _line.partition("=")
            os.environ.setdefault(_key.strip(), _val.strip().strip('"'))
    print("[INFO] dev.env loaded")
else:
    print("[WARN] dev.env not found — relying on existing environment variables")

[INFO] dev.env loaded


In [9]:
ks_list = [5, 10, 20]
top_k = 20
binary_threshold = None

CLOUDRUN_URL = os.getenv("CLOUDRUN_URL", "http://localhost:8001")

evaluator = SearchEvaluator(ks=ks_list)

In [11]:
import requests

# Health-check the Cloud Run embedding service before running benchmarks
try:
    resp = requests.get(f"{CLOUDRUN_URL}/health", timeout=15)
    resp.raise_for_status()
    h = resp.json()
    print("[OK] Embedding service is healthy")
    print(f"     device  : {h.get('device')}")
    print(f"     encoder : {h.get('encoder')}")
    print(f"     reranker: {h.get('reranker')}")
except Exception as e:
    print(f"[WARN] Could not reach embedding service: {e}")

inference_client = OpenSearchInference(
    bm25_index="bm25_index",
    hnsw_index="hnsw_index",
    embedding_service_url=CLOUDRUN_URL,
)

[OK] Embedding service is healthy
     device  : cuda
     encoder : /models/finetuned_encoder
     reranker: /models/bge-reranker-v2-m3


In [12]:
# load datasets
with open(processed_dir / "test_qrels.json", "r", encoding="utf-8") as f:
    gt_qrels_raw = json.load(f)

test_queries = pd.read_parquet(processed_dir / "test_query_table.parquet")

### BM25 Benchmarking

In [18]:
bm25_ranked = retrieve_bm25(
    inference_client,
    query_table=test_queries,
    topn=top_k,
    filter_source=None,
)

BM25 retrieval: 100%|██████████| 9100/9100 [00:17<00:00, 510.56it/s]


In [19]:
bm25_scores = evaluator.evaluate_rankings(
    ground_truth_qrels=gt_qrels_raw,
    predicted_rankings=bm25_ranked,
    binary_threshold=binary_threshold,
)

In [20]:
bm25_scores

,metric,k,score
0,mrr,5,0.753833
1,mrr,10,0.756222
2,mrr,20,0.759065
3,ndcg,5,0.585556
4,ndcg,10,0.564600
5,ndcg,20,0.548397
6,recall,5,0.152629
7,recall,10,0.263160
8,recall,20,0.389838


### HNSW Benchmarking

### HNSW Benchmarking (Finetuned Encoder via Cloud Run)

In [22]:
hnsw_ranked = retrieve_hnsw(
    inference_client,
    query_table=test_queries,
    topn=top_k,
    filter_source=None,
)

[HNSW] Batch-encoding 9100 queries via embedding service...


HNSW retrieval: 100%|██████████| 9100/9100 [00:34<00:00, 261.34it/s]


In [23]:
hnsw_scores = evaluator.evaluate_rankings(
    ground_truth_qrels=gt_qrels_raw,
    predicted_rankings=hnsw_ranked,
    binary_threshold=binary_threshold,
)

In [24]:
hnsw_scores

,metric,k,score
0,mrr,5,0.783583
1,mrr,10,0.788645
2,mrr,20,0.790411
3,ndcg,5,0.601717
4,ndcg,10,0.585437
5,ndcg,20,0.580953
6,recall,5,0.159501
7,recall,10,0.280364
8,recall,20,0.425425


### Hybrid (BM25 + HNSW) with RRF Benchmarking

In [25]:
# inference_client already has the finetuned encoder + reranker loaded at setup.
print("[INFO] Using existing inference_client (finetuned encoder + reranker)")

hybrid_ranked = retrieve_hybrid_rrf(
    inference_client,
    query_table=test_queries,
    topn=top_k,
    filter_source=None,
    candidate_pool_size=30,
)

[INFO] Using existing inference_client (finetuned encoder + reranker)
[Hybrid RRF] Batch-encoding 9100 queries via embedding service...


Hybrid RRF retrieval: 100%|██████████| 9100/9100 [01:03<00:00, 144.38it/s]


In [26]:
hybrid_rrf_scores = evaluator.evaluate_rankings(
    ground_truth_qrels=gt_qrels_raw,
    predicted_rankings=hybrid_ranked,
    binary_threshold=binary_threshold,
)

In [27]:
hybrid_rrf_scores

,metric,k,score
0,mrr,5,0.806583
1,mrr,10,0.811692
2,mrr,20,0.813273
3,ndcg,5,0.635354
4,ndcg,10,0.617524
5,ndcg,20,0.603703
6,recall,5,0.166494
7,recall,10,0.295286
8,recall,20,0.444602


### Hybrid (BM25 + HNSW) with Reranking Model Benchmarking

In [13]:
# ── Stratified 1,000-query sample for reranker benchmarking ──────────────────
# Reranking via Cloud Run (~4 s/query) would take ~10 h on all 9,100 queries.
# A 1,000-query stratified sample gives stable NDCG estimates in ~1 h while
# preserving the ESCI/WANDS source distribution.

SAMPLE_SIZE = 1000
RANDOM_SEED = 42

if "source" in test_queries.columns:
    # Proportional stratified sample: each source keeps its share of SAMPLE_SIZE
    sample_queries = (
        test_queries
        .groupby("source", group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, round(SAMPLE_SIZE * len(g) / len(test_queries))),
            random_state=RANDOM_SEED,
        ))
        .sample(frac=1, random_state=RANDOM_SEED)   # shuffle rows
        .reset_index(drop=True)
    )
else:
    sample_queries = (
        test_queries
        .sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

# Filter ground-truth qrels to sampled query IDs only
sampled_qids = set(sample_queries["query_id"].astype(str))
sample_qrels = {qid: v for qid, v in gt_qrels_raw.items() if qid in sampled_qids}

print(f"Sample : {len(sample_queries):,} queries  |  qrels: {len(sample_qrels):,}")
if "source" in sample_queries.columns:
    print(sample_queries["source"].value_counts().to_string())

Sample : 1,000 queries  |  qrels: 27


In [14]:
import time

t0 = time.perf_counter()

hybrid_reranking_ranked = retrieve_hybrid_rerank(
    inference_client,
    query_table=sample_queries,
    topn=top_k,
    filter_source=None,
    candidate_pool_size=10,
)

elapsed = time.perf_counter() - t0
print(f"Completed {len(sample_queries)} queries in {elapsed/60:.1f} min  ({elapsed/len(sample_queries):.2f} s/query)")

[Hybrid Rerank] Batch-encoding 1000 queries via embedding service...


Hybrid Rerank retrieval: 100%|██████████| 1000/1000 [22:42<00:00,  1.36s/it]

Completed 1000 queries in 22.7 min  (1.36 s/query)


In [15]:
hybrid_reranker_scores = evaluator.evaluate_rankings(
    ground_truth_qrels=sample_qrels,
    predicted_rankings=hybrid_reranking_ranked,
    binary_threshold=binary_threshold,
)

In [16]:
hybrid_reranker_scores

,metric,k,score
0,mrr,5,0.822222
1,mrr,10,0.822222
2,mrr,20,0.822222
3,ndcg,5,0.688279
4,ndcg,10,0.645326
5,ndcg,20,0.558813
6,recall,5,0.183621
7,recall,10,0.318305
8,recall,20,0.396521


## Results Comparison — All Methods

In [28]:
def _ndcg_row(scores_df: pd.DataFrame, method: str) -> dict:
    ndcg = (
        scores_df[scores_df["metric"] == "ndcg"]
        .set_index("k")["score"]
    )
    return {
        "Method": method,
        "NDCG@5":  round(ndcg[5],  4),
        "NDCG@10": round(ndcg[10], 4),
        "NDCG@20": round(ndcg[20], 4),
    }

comparison = pd.DataFrame([
    _ndcg_row(bm25_scores,              "BM25"),
    _ndcg_row(hnsw_scores,              "HNSW (Finetuned Encoder)"),
    _ndcg_row(hybrid_rrf_scores,        "Hybrid RRF (BM25 + HNSW finetuned)"),
    _ndcg_row(hybrid_reranker_scores,   "Hybrid + Cross-encoder Reranker"),
]).sort_values("NDCG@5", ascending=False).reset_index(drop=True)

comparison.insert(0, "Rank", range(1, len(comparison) + 1))

comparison.style \
    .format({"NDCG@5": "{:.4f}", "NDCG@10": "{:.4f}", "NDCG@20": "{:.4f}"}) \
    .background_gradient(subset=["NDCG@5", "NDCG@10", "NDCG@20"], cmap="RdYlGn") \
    .set_properties(**{"text-align": "left"}) \
    .hide(axis="index")

Rank,Method,NDCG@5,NDCG@10,NDCG@20
1,Hybrid + Cross-encoder Reranker,0.6883,0.6453,0.5588
2,Hybrid RRF (BM25 + HNSW finetuned),0.6354,0.6175,0.6037
3,HNSW (Finetuned Encoder),0.6017,0.5854,0.5810
4,BM25,0.5856,0.5646,0.5484


## Benchmark Results

Four retrieval architectures were evaluated using **Normalised Discounted Cumulative Gain (NDCG)** at cutoffs $k \in \{5, 10, 20\}$, defined as:

$$\text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}, \qquad \text{DCG@}k = \sum_{i=1}^{k} \frac{2^{r_i} - 1}{\log_2(i + 1)}$$

| Rank | Method | NDCG@5 | NDCG@10 | NDCG@20 |
|------|--------|--------|---------|---------|
| 🥇 1 | **Hybrid + Cross-encoder Reranker** | **0.6883** | **0.6453** | 0.5588 |
| 2 | Hybrid RRF (BM25 + HNSW Finetuned) | 0.6354 | 0.6175 | **0.6037** |
| 3 | HNSW (Finetuned Encoder) | 0.6017 | 0.5854 | 0.5810 |
| 4 | BM25 | 0.5856 | 0.5646 | 0.5484 |

### Winning Architecture

The **Hybrid + Cross-encoder Reranker** pipeline dominates at shallow cutoffs — the operationally critical regime for e-commerce search where users rarely scroll past the first 5–10 results:

$$\Delta\text{NDCG@5}_{\text{vs BM25}} = \frac{0.6883 - 0.5856}{0.5856} \approx +17.6\%$$

$$\Delta\text{NDCG@5}_{\text{vs Hybrid RRF}} = \frac{0.6883 - 0.6354}{0.6354} \approx +8.3\%$$

### Key Finding: Rank Inversion at NDCG@20

A noteworthy inversion occurs at the deeper cutoff — **Hybrid RRF overtakes the reranker** at NDCG@20 ($0.6037 > 0.5588$). This is expected behaviour: the cross-encoder reranks a fixed candidate pool of size $C$ drawn from BM25 $\cup$ HNSW, so its influence is bounded by retrieval recall. Beyond the reranked window, ordering reverts to the fused score, degrading gracefully. Hybrid RRF, by contrast, maintains consistent rank fusion signal across the full depth $k = 20$.

> **Conclusion:** For a production ranking objective prioritising top-$k$ precision ($k \leq 10$), the Hybrid + Cross-encoder Reranker is the clear winner. For recall-oriented use cases (broad candidate generation, offline indexing), Hybrid RRF is preferable.